In [1]:
import os
import glob
import ee
import geemap
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

# 1. Inicialização do GEE (High-Volume API para downloads mais rápidos)
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

# 2. Obter o Bounding Box de TODO o GeoDataFrame
path_json = "../data/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)

# Pega o envelope retangular da área inteira [minx, miny, maxx, maxy]
total_bounds = gdf.total_bounds.tolist()
area_bbox = ee.Geometry.Rectangle(total_bounds)

# 3. Processamento dos Rasters no GEE
modis = (
    ee.ImageCollection('MODIS/061/MOD09A1')
    .filterBounds(area_bbox)
    .filterDate('2000-02-18', '2025-12-31')
)

def add_indices(image):
    img_scaled = image.select('sur_refl_b.*').multiply(0.0001)
    
    # MSAVI
    msavi = img_scaled.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2',
        {
            'NIR': img_scaled.select('sur_refl_b02'),
            'RED': img_scaled.select('sur_refl_b01')
        }
    ).rename('MSAVI')

    # Albedo
    albedo = img_scaled.expression(
        '0.160 * B1 + 0.291 * B2 + 0.243 * B3 + 0.116 * B4 + 0.112 * B5 + 0.081 * B7 - 0.0015',
        {
            'B1': img_scaled.select('sur_refl_b01'),
            'B2': img_scaled.select('sur_refl_b02'),
            'B3': img_scaled.select('sur_refl_b03'),
            'B4': img_scaled.select('sur_refl_b04'),
            'B5': img_scaled.select('sur_refl_b05'),
            'B7': img_scaled.select('sur_refl_b07')
        }
    ).rename('Albedo')

    # Retorna apenas as duas bandas calculadas
    return image.addBands([msavi, albedo]).select(['MSAVI', 'Albedo']).copyProperties(image, ['system:time_start'])


In [5]:
modis_indices = modis.map(add_indices)

# 5. GERAR MEDIANA ANUAL NO GEE
anos = ee.List.sequence(2000, 2025)

def criar_mediana_anual(ano):
    ano = ee.Number(ano)
    data_inicio = ee.Date.fromYMD(ano, 1, 1)
    data_fim = ee.Date.fromYMD(ano, 12, 31)
    
    mediana = (
        modis_indices
        .filterDate(data_inicio, data_fim)
        .median() # Reduz todas as imagens do ano para a mediana pixel a pixel
        .clip(area_bbox)
    )
    
    # Define as propriedades de tempo para o xarray reconhecer o ano
    return mediana.set({
        'year': ano,
        'system:time_start': data_inicio.millis(),
        'system:index': ano.format('%d')
    })

# Transforma a lista de anos em uma nova ImageCollection de 26 imagens
colecao_anual = ee.ImageCollection(anos.map(criar_mediana_anual))

In [14]:
# 6. Download dos 26 Rasters Anuais para o SSD
out_dir = "../data/modis_medianas_anuais_tif"
os.makedirs(out_dir, exist_ok=True)

print("Iniciando o download das 26 medianas anuais (2000-2025)...")
geemap.ee_export_image_collection(
    colecao_anual,
    out_dir=out_dir,
    scale=1000,               # 500 metros,   
    region=area_bbox,
    file_per_band=False,
)

print("Download concluído! Criando o cubo Zarr local...")

Iniciando o download das 26 medianas anuais (2000-2025)...
Total number of images: 26

Exporting 1/26: ../data/modis_medianas_anuais_tif/2000.tif
Generating URL ...
Please wait ...
Data downloaded to /home/usuario/Área de trabalho/desertificacao_sf/data/modis_medianas_anuais_tif/2000.tif


Exporting 2/26: ../data/modis_medianas_anuais_tif/2001.tif
Generating URL ...
Please wait ...
Data downloaded to /home/usuario/Área de trabalho/desertificacao_sf/data/modis_medianas_anuais_tif/2001.tif


Exporting 3/26: ../data/modis_medianas_anuais_tif/2002.tif
Generating URL ...
Please wait ...
Data downloaded to /home/usuario/Área de trabalho/desertificacao_sf/data/modis_medianas_anuais_tif/2002.tif


Exporting 4/26: ../data/modis_medianas_anuais_tif/2003.tif
Generating URL ...
Please wait ...
Data downloaded to /home/usuario/Área de trabalho/desertificacao_sf/data/modis_medianas_anuais_tif/2003.tif


Exporting 5/26: ../data/modis_medianas_anuais_tif/2004.tif
Generating URL ...
Please wait ...
Dat